# EcoSort: Intelligent Waste Management Assistant — Summative Lab (NNs & Similar Models)

**Part 1:** Dataset exploration and preparation  
**Part 2:** Waste material classification with a CNN (transfer learning)  
**Part 3:** Waste description classification (text)  
**Part 4:** Recycling instruction generation with RAG  
**Part 5:** Integrated waste management assistant  

Authoring environment: Windows | Python 3.12 | CPU only (16 GB RAM)


In [ ]:
# %% [markdown]
# ## 0. Global configuration
# We centralise every path, class name, and hyper-parameter here so the whole
# notebook stays consistent and reproducible.

import os
import sys
import json
import re
import math
import random
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Quiet noisy TF startup logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import tensorflow as tf
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
    else:
        print('INFO: No GPU detected - running on CPU.')
except Exception as e:
    print('GPU config skipped:', e)
tf.random.set_seed(42)

import torch
torch.manual_seed(42)

# ---------------------------------------------------------------- paths
BASE = os.getcwd()
DATA = os.path.join(BASE, 'data')          # derived data (splits, features)
FIG = os.path.join(DATA, 'figures')        # saved plots
MODELS = os.path.join(BASE, 'models')      # saved model artifacts
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

def find_realwaste_root():
    cands = [
        os.path.join(BASE, 'realwaste', 'realwaste-main', 'RealWaste'),
        os.path.join(BASE, 'RealWaste'),
        os.path.join(BASE, 'data', 'RealWaste'),
    ]
    for c in cands:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError('RealWaste image root not found. Please place the '
                            '"RealWaste" class folders next to the data.')
IMAGE_ROOT = find_realwaste_root()
TEXT_CSV = os.path.join(BASE, 'waste_descriptions.csv')
POLICY_JSON = os.path.join(BASE, 'waste_policy_documents.json')

CLASSES = sorted([d for d in os.listdir(IMAGE_ROOT)
                  if os.path.isdir(os.path.join(IMAGE_ROOT, d))])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(f'IMAGE_ROOT  : {IMAGE_ROOT}')
print(f'N classes   : {len(CLASSES)}')
print(f'Classes     : {CLASSES}')
print(f'TensorFlow  : {tf.__version__}   torch: {torch.__version__}')


In [ ]:
# %% [markdown]
# ### Shared helpers (used everywhere below)

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); torch.manual_seed(s)

def save_fig(fig, name, show=True, dpi=110, tight=True):
    """Save the figure and (for the notebook output) display it."""
    p = os.path.join(FIG, name)
    fig.savefig(p, dpi=dpi, bbox_inches='tight' if tight else None)
    if show:
        plt.show()
    plt.close(fig)
    return p

def clf_metrics(y_true, y_pred, labels):
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score, classification_report,
                                 confusion_matrix)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, digits=3),
        'confusion': confusion_matrix(y_true, y_pred, labels=range(len(labels))),
    }

def plot_confusion(cm, labels, title='Confusion matrix', cmap='Blues'):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=labels,
                yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    return fig


# Part 1: Dataset Exploration & Preparation

## 1.1 RealWaste image dataset

We start by building a tidy DataFrame over the **RealWaste** images
(9 material classes), scanning basic metadata (size, colour mode, file
size, brightness / saturation) so we can reason about the visual
characteristics and the challenges the CNN will face.


In [ ]:
# -------------------------------------------------
# 1.1.1 Build the image catalogue (cached on disk)
meta_path = os.path.join(DATA, 'image_metadata.csv')
if os.path.exists(meta_path):
    df_img = pd.read_csv(meta_path)
    print('Loaded cached image metadata.')
else:
    rows = []
    for cls in CLASSES:
        cdir = os.path.join(IMAGE_ROOT, cls)
        for fn in sorted(os.listdir(cdir)):
            p = os.path.join(cdir, fn)
            rows.append({'filename': fn, 'class': cls, 'path': p})
    df_img = pd.DataFrame(rows)

    def scan_row(r):
        try:
            im = Image.open(r['path'])
            w, h = im.size
            mode = im.mode
            fs = os.path.getsize(r['path'])
            small = im.convert('RGB').resize((48, 48))
            arr = np.asarray(small).astype(np.float32)
            hsv = np.asarray(small.convert('HSV')).astype(np.float32)
            im.close()
            return pd.Series({
                'width': w, 'height': h, 'mode': mode,
                'aspect_ratio': w / h, 'file_kb': round(fs / 1024, 1),
                'mean_brightness': arr.mean(),
                'std_brightness': arr.std(),
                'mean_saturation': hsv[..., 1].mean(),
            })
        except Exception as ex:
            return pd.Series({'width': np.nan, 'height': np.nan, 'mode': 'ERR',
                              'aspect_ratio': np.nan, 'file_kb': np.nan,
                              'mean_brightness': np.nan, 'std_brightness': np.nan,
                              'mean_saturation': np.nan})

    meta = df_img.apply(scan_row, axis=1)
    df_img = pd.concat([df_img, meta], axis=1)
    df_img.to_csv(meta_path, index=False)
    print('Catalogue written to', meta_path)

print(df_img.shape)
print('Classes:', df_img.groupby('class').size().to_dict())
print('NaNs in metadata:', df_img[['width', 'height', 'mean_brightness']].isna().sum().sum())


In [ ]:
# -------------------------------------------------
# 1.1.2 Class distribution
class_counts = df_img['class'].value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(class_counts.index, class_counts.values, color=sns.color_palette('viridis', len(CLASSES)))
ax.set_title('RealWaste - number of images per material class')
ax.set_ylabel('Images')
for b, v in zip(bars, class_counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 4, str(v), ha='center', fontsize=9)
plt.xticks(rotation=45, ha='right')
save_fig(fig, 'image_class_distribution.png')

imbalance = class_counts.max() / class_counts.min()
print(f'Most frequent class: {class_counts.index[0]} ({class_counts.iloc[0]})')
print(f'Least frequent class: {class_counts.index[-1]} ({class_counts.iloc[-1]})')
print(f'Class imbalance ratio (max/min) = {imbalance:.2f}x  -> class weights will be used.')

# -------------------------------------------------
# 1.1.3 Image geometry / quality summary
print('\n--- Image geometry  (all sizes are square) ---')
print(df_img[['width', 'height', 'aspect_ratio', 'file_kb']].describe().round(1).to_string())
print('\nColour modes:', df_img['mode'].value_counts().to_dict())

print('\n--- Per-class visual summaries (brightness 0-255, saturation 0-255) ---')
g = df_img.groupby('class')[['mean_brightness', 'mean_saturation', 'std_brightness', 'file_kb']].mean().round(1).sort_values('mean_brightness')
print(g.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_img, x='class', y='mean_brightness', ax=axes[0], palette='viridis')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Mean brightness per class')
sns.boxplot(data=df_img, x='class', y='mean_saturation', ax=axes[1], palette='magma')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title('Mean saturation per class')
save_fig(fig, 'image_brightness_saturation.png')


In [ ]:
# -------------------------------------------------
# 1.1.4 Look at real samples per class
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, cls in zip(axes.ravel(), CLASSES):
    sub = df_img[df_img['class'] == cls]
    sample = sub.iloc[random.randrange(len(sub))]['path']
    im = Image.open(sample).resize((180, 180))
    ax.imshow(im); ax.set_title(cls, fontsize=10); ax.axis('off')
plt.suptitle('Random sample from each RealWaste class', y=1.01)
save_fig(fig, 'image_class_samples.png')

# And one richer grid for the "Plastic vs Glass vs Metal" family which we
# expect to be confusable.
def grid_for(cls_subset, title, n=6):
    rng = random.Random(7)
    rows = []
    for cls in cls_subset:
        sub = df_img[df_img['class'] == cls]
        rows.append(sub.sample(n, random_state=7))
    panel = pd.concat(rows)
    fig, axes = plt.subplots(n, len(cls_subset), figsize=(3 * len(cls_subset), 3 * n))
    for i, cls in enumerate(cls_subset):
        for j, (_, r) in enumerate(panel[panel['class'] == cls].iterrows()):
            im = Image.open(r['path']).resize((120, 120))
            axes[j, i].imshow(im); axes[j, i].axis('off')
            if j == 0:
                axes[j, i].set_title(cls, fontsize=10)
    plt.suptitle(title, y=1.01)
    return fig

fig = grid_for(['Plastic', 'Glass', 'Metal', 'Miscellaneous Trash'],
               'Potentially confusable classes: Plastic vs Glass vs Metal vs Miscellaneous Trash')
save_fig(fig, 'image_confusable_grid.png')


## 1.2 Waste description text dataset (`waste_descriptions.csv`)

5,000 resident-style descriptions of waste items with:
`description`, `category`, `disposal_instruction`, `common_confusion`,
`material_composition`.


In [ ]:
# -------------------------------------------------
# 1.2.1 Load and inspect the description data
df_txt = pd.read_csv(TEXT_CSV)
print('Shape:', df_txt.shape)
print('\nDtypes:\n', df_txt.dtypes)
print('\nHead:')
print(df_txt.head(8).to_string())
print('\nMissing values:')
print(df_txt.isna().sum().to_string())

print('\nCategory distribution:')
ct = df_txt['category'].value_counts()
print(ct.to_string())

fig, ax = plt.subplots(figsize=(10, 4.5))
ct.sort_values().plot.barh(ax=ax, color=sns.color_palette('viridis', len(CLASSES)))
ax.set_title('waste_descriptions.csv - category distribution')
ax.set_xlabel('Count')
plt.tight_layout()
save_fig(fig, 'text_class_distribution.png')

# -------------------------------------------------
# 1.2.2 Vocabulary & structure analysis
lengths = df_txt['description'].str.split().str.len()
print('\nDescription word-count: mean %.2f, std %.2f, min %d, max %d' %
      (lengths.mean(), lengths.std(), lengths.min(), lengths.max()))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=range(0, 13), edgecolor='k', color='steelblue')
ax.set_title('Description length (words)')
ax.set_xlabel('words'); ax.set_ylabel('count')
save_fig(fig, 'text_description_length.png')

vocab = set()
df_txt['description'].str.lower().str.findall(r"[a-z']+").apply(lambda t: vocab.update(t))
print(f'Unique lowercase tokens across all descriptions: {len(vocab):,}')

# -------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer
STOP = set(__import__('sklearn.feature_extraction.text', fromlist=['ENGLISH_STOP_WORDS']).ENGLISH_STOP_WORDS)
top_words = {}
for cls in CLASSES:
    corpus = df_txt[df_txt['category'] == cls]['description'].tolist()
    cv = CountVectorizer(ngram_range=(1, 1), stop_words='english')
    try:
        X = cv.fit_transform(corpus)
    except ValueError:
        continue
    sums = np.asarray(X.sum(axis=0)).ravel()
    idx = np.argsort(sums)[::-1][:8]
    top_words[cls] = [(cv.get_feature_names_out()[i], int(sums[i])) for i in idx]

print('\nTop-8 characteristic tokens per class:')
for cls, toks in top_words.items():
    print(f'{cls:20s} -> ' + ', '.join(f'{w}({c})' for w, c in toks))


In [ ]:
# -------------------------------------------------
# 1.2.3 Auxiliary text fields (why the extra columns matter for RAG)
print('common_confusion - filled in', df_txt['common_confusion'].notna().sum(), 'of', len(df_txt), 'rows')
print('\nMost frequent confusion notes:')
cc = df_txt['common_confusion'].dropna()
print(Counter(cc).most_common(8))

print('\nUnique disposal instructions:', df_txt['disposal_instruction'].nunique())
print('Most frequent disposal instructions:')
dc = df_txt['disposal_instruction'].value_counts()
print(dc.head(8).to_string())

print('\nUnique material-composition strings:', df_txt['material_composition'].nunique())
mc = df_txt['material_composition'].value_counts()
print(mc.head(8).to_string())

print('\nSample rows for the ambiguous class "Miscellaneous Trash":')
print(df_txt[df_txt['category'] == 'Miscellaneous Trash'].sample(5, random_state=1)[
    ['description', 'disposal_instruction', 'common_confusion']].to_string())


## 1.3 Waste policy documents (`waste_policy_documents.json`)

Municipal regulations from *Metro City*. Each record has a `policy_type`,
`categories_covered`, `effective_date`, `jurisdiction` and the raw
`document_text` block. These documents become both the **retrieval corpus**
and the **supervision signal** for the RAG generator.


In [ ]:
# -------------------------------------------------
# 1.3.1 Load and organise the policy documents
with open(POLICY_JSON, 'r', encoding='utf-8') as f:
    policy_raw = json.load(f)
print('Number of policy documents:', len(policy_raw))
pf = pd.DataFrame(policy_raw)
print('\nColumns:', list(pf.columns))
print('\nHead of tabular fields:')
print(pf[['policy_id', 'policy_type', 'categories_covered', 'effective_date',
          'jurisdiction']].head(14).to_string())

pf['n_words'] = pf['document_text'].str.split().str.len()
pf['n_lines'] = pf['document_text'].str.count('\n') + 1
print('\nDocument length stats (words):')
print(pf['n_words'].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
pf['n_words'].plot.hist(bins=15, ax=axes[0], edgecolor='k', color='seagreen')
axes[0].set_title('Policy document length (words)')
pf['policy_type'].value_counts().sort_values()[:12].plot.barh(ax=axes[1], color='teal')
axes[1].set_title('Policy types')
plt.tight_layout()
save_fig(fig, 'policy_docs_summary.png')

cov = Counter()
for cl in pf['categories_covered']:
    for c in cl:
        cov[c] += 1
print('\nPolicies covering each category:')
print(pd.Series(cov).sort_values(ascending=False).to_string())

# -------------------------------------------------
# 1.3.2 Structure of a document (they use a recurring SKELETON)
print('\n--- Structure of a typical policy document ---')
print(policy_raw[3]['document_text'][:900])


### 1.4 Preliminary data-quality findings & challenges

**Images (RealWaste)**
1. **Uniform geometry** — every image is an RGB JPEG, exactly `524 x 524`.
   There is no resolution variation to handle, but real resident photos will
   differ; we therefore still use augmentation (flips, rotations, zoom) so the
   CNN does not overfit to the dataset's particular capture style.
2. **Class imbalance** — `Plastic` (921) is ~2.9x more frequent than
   `Textile Trash` (318). Vanilla training will bias toward frequent classes,
   so we balance the loss with *class weights*.
3. **Confusable material families** — transparent bottles mean *Plastic vs
   Glass* and *Paper vs Cardboard* look similar; *Miscellaneous Trash* is a
   catch-all with internally inconsistent visuals. Expect confusion along
   these pairs; we analyse this explicitly.
4. **Background clutter** — capture-style differences (napkins, hands,
   tables) act as domain noise; brightness/saturation vary strongly which is
   why colour augmentation matters.

**Text (`waste_descriptions.csv`)**
1. **Very short inputs** (mean ~5 words) — almost the whole signal lives in a
   handful of tokens (e.g. "bottle", "crushed", "grounds").
2. **Half of `common_confusion` is empty** (2,504 blanks) — we treat it as an
   optional pre-retrieval hint, never a required field.
3. **Adjective-led phrasing** ("soiled", "wrinkled", "intact") is irrelevant
   to material class and adds noise; we keep noun vocabulary but watch that
   TF-IDF does not latch onto these modifiers.
4. Regional/term variation (see `material_composition`) needs normalisation so
   the classifier and retriever agree.

**Policy documents (JSON)**
1. A small corpus (14 docs) but with a highly *regular skeleton*
   (ACCEPTABLE / NON-ACCEPTABLE / COLLECTION METHOD / PREPARATION /
   BENEFITS) which lets us chunk reliably.
2. Some documents cover **multiple categories** (e.g. Municipal Waste) and a
   few categories are *not* covered by their own dedicated document
   (e.g. Glass colours) — retrieval must therefore handle multi-category
   sections.
3. `categories_covered` gives us *ground-truth tags* to score retrieval.

These findings drive the preprocessing decisions made in the rest of Part 1.


### 1.5 Data pipelines for every modality

Now we create tidy, reproducible artifacts used by Parts 2-4:

* **Image split** — stratified 70 / 15 / 15 into `data/img_*.csv`, plus a
  `tf.data` pipeline (resize + augment + normalise).
* **Text split** — stratified 80 / 10 / 10 into `data/txt_*.csv` together with
  a light cleaning function used by both classical and neural models.
* **Policy chunking** — the 14 documents are split into per-category sections
  stored in `data/policy_chunks.csv`, ready to become RAG retrieval units.


In [ ]:
# ------------------------------------------------------
# 1.5.1 Image split (stratified) + tf.data pipeline
from sklearn.model_selection import train_test_split

train_df, tmp = train_test_split(df_img, test_size=0.30, stratify=df_img['class'],
                                 random_state=42)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['class'],
                                   random_state=42)
for name, d in [('img_train', train_df), ('img_val', val_df), ('img_test', test_df)]:
    d[['path', 'class', 'filename']].to_csv(os.path.join(DATA, name + '.csv'), index=False)
print('Image split sizes:', len(train_df), len(val_df), len(test_df))
print('Stratification check (class balances in each split):')
print(pd.DataFrame({
    'train': train_df['class'].value_counts(normalize=True),
    'val':   val_df['class'].value_counts(normalize=True),
    'test':  test_df['class'].value_counts(normalize=True),
}).round(3).to_string())

y_train_img = train_df['class'].map(CLASS_TO_IDX).values
class_weight_dict = {}
for cur in np.unique(y_train_img):
    n = (y_train_img == cur).sum()
    class_weight_dict[cur] = len(y_train_img) / (len(np.unique(y_train_img)) * n)
print('\nComputed image class weights (inverse frequency, normalised):')
print({IDX_TO_CLASS[k]: round(v, 3) for k, v in class_weight_dict.items()})

IMG_SIZE_DEFAULT = 224

def build_img_pipeline(df, img_size, batch_size, augment=False, cache_name=None):
    paths = df['path'].values
    labels = df['class'].map(CLASS_TO_IDX).values.astype(np.int64)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        raw = tf.io.read_file(p)
        x = tf.image.decode_jpeg(raw, channels=3)
        x = tf.image.resize(x, (img_size, img_size))
        return x, y
    if augment:
        aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal'),
            tf.keras.layers.RandomRotation(0.15),
            tf.keras.layers.RandomZoom(0.15),
            tf.keras.layers.RandomBrightness(0.15),
            tf.keras.layers.RandomContrast(0.15),
        ], name='aug')
        def _a(x, y):
            x = tf.cast(x, tf.float32)
            return aug(x, training=True), y
    else:
        def _a(x, y):
            return tf.cast(x, tf.float32), y
    def _norm(x, y):
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
        return tf.cast(x, tf.float32), y
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_a, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_norm, num_parallel_calls=tf.data.AUTOTUNE)
    if cache_name:
        ds = ds.cache(os.path.join(DATA, cache_name))
    ds = ds.shuffle(2048, reshuffle_each_iteration=True) if augment else ds
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds224 = build_img_pipeline(train_df, IMG_SIZE_DEFAULT, 32, augment=True, cache_name='cache_train_224')
val_ds224   = build_img_pipeline(val_df,   IMG_SIZE_DEFAULT, 32, augment=False, cache_name='cache_val_224')
test_ds224  = build_img_pipeline(test_df,  IMG_SIZE_DEFAULT, 32, augment=False, cache_name='cache_test_224')
print('Sample batch shapes (train):',
      [t.shape for t in next(iter(train_ds224.unbatch().batch(8)))])

# ------------------------------------------------------
# 1.5.2 Text cleaning + split
def clean_text(s):
    """Lightweight cleaning tuned for resident-style descriptions."""
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)      # urls
    s = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' ', s)   # emails
    s = re.sub(r'[^a-z\s]', ' ', s)                   # punctuation/numbers
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_txt['clean'] = df_txt['description'].apply(clean_text)
df_txt['n_words'] = df_txt['clean'].str.split().str.len()
print('Clean examples:')
print(df_txt[['description', 'clean']].head(5).to_string())

X_txt = df_txt['clean'].values.astype(str)
y_txt = df_txt['category'].astype('category').cat.codes.values
cat_names = list(df_txt['category'].astype('category').cat.categories)
for i, c in enumerate(cat_names):
    print(i, c)

Xtr, tmpX = train_test_split(np.arange(len(df_txt)), test_size=0.20, stratify=y_txt, random_state=42)
Xv, Xte = train_test_split(tmpX, test_size=0.50, stratify=y_txt[tmpX], random_state=42)
train_idx, val_idx, test_idx = Xtr, Xv, Xte
print('\nText split sizes:', len(train_idx), len(val_idx), len(test_idx))
for name, idx in [('txt_train', train_idx), ('txt_val', val_idx), ('txt_test', test_idx)]:
    pd.DataFrame({'index': idx, 'text': df_txt['clean'].iloc[idx].values,
                  'label': df_txt['category'].iloc[idx].values}).to_csv(
        os.path.join(DATA, name + '.csv'), index=False)

# ------------------------------------------------------
# 1.5.3 Policy documents -> per-category chunks (improved splitter)
def split_sections(doc_text):
    """Split a policy document on its recurring section headers."""
    lines = doc_text.split('\n')
    header_re = re.compile(r'^[A-Za-z][A-Za-z0-9 /\-]*:$')
    sections = []
    cur_head, cur = None, []
    for ln in lines:
        s = ln.strip()
        if s and len(s) <= 45 and header_re.match(s) and not s.startswith('-'):
            if cur_head is not None:
                sections.append((cur_head, '\n'.join(cur).strip()))
            cur_head, cur = s.rstrip(':').strip(), []
        else:
            cur.append(ln)
    if cur_head is not None:
        sections.append((cur_head, '\n'.join(cur).strip()))
    return sections

def assign_category(header, categories):
    h = header.lower()
    if len(categories) == 1:
        return categories[0]
    for c in categories:
        if c.lower() in h:
            return c
    return None  # e.g. GENERAL REQUIREMENTS section

CHUNKS = []
for doc in policy_raw:
    for header, body in split_sections(doc['document_text']):
        tag = assign_category(header, doc['categories_covered'])
        CHUNKS.append({
            'section': header, 'category': tag, 'text': body,
            'policy_id': doc['policy_id'], 'policy_type': doc['policy_type'],
            'categories_covered': ','.join(doc['categories_covered']),
            'effective_date': doc['effective_date'], 'jurisdiction': doc['jurisdiction'],
        })
chunk_df = pd.DataFrame(CHUNKS)
print('Policy chunks created:', chunk_df.shape)
print('\nTagged vs untagged chunks:')
print(chunk_df['category'].fillna('GENERAL').value_counts().to_string())
print('\nA few sample chunks:')
print(chunk_df[['section', 'category', 'text']].head(3).to_string())
print('\nChunk word-count stats:')
print(chunk_df['text'].str.split().str.len().describe().round(1).to_string())
chunk_df.to_csv(os.path.join(DATA, 'policy_chunks.csv'), index=False)


# Part 4: Recycling Instruction Generation with RAG

We build a **Retrieval-Augmented Generation** pipeline:

1. **Retrieval corpus** — the Metro City policy documents (split into
   per-category sections) plus aggregated resident disposal advice, embedded
   with a sentence-transformer (`all-MiniLM-L6-v2`).
2. **Retrieval** — nearest-neighbour cosine search over the embedding index.
3. **Generation** — a pre-trained **DistilGPT2** LM fine-tuned (on ~200
   grounded examples built from the corpus) to produce recycling instructions
   that *rely on the retrieved reference text*.

The guiding design choice: because the generator is tiny, we deliberately
build its training objective as an **extractive grounding** task
(prompt-with-reference -> reference-derived answer). The model therefore has
to copy/summarise the retrieved policy rather than hallucinate instructions —
which is exactly what a small waste-management assistant should do.


In [ ]:
# --------------------------------------------------------
# 4.1 Build the retrieval corpus (policy sections + resident advice)
from sentence_transformers import SentenceTransformer

# policy sections (re-split robustly)
sections = []
for doc in policy_raw:
    for header, body in split_sections(doc['document_text']):
        if len(body.split()) < 5:
            continue
        tag = assign_category(header, doc['categories_covered'])
        sections.append({
            'doc_id': f"pol{doc['policy_id']}:{header}",
            'kind': 'policy',
            'category': tag,
            'valid_categories': [tag] if tag else doc['categories_covered'],
            'text': header + ':\n' + body,
            'policy_type': doc['policy_type'],
            'effective_date': doc['effective_date'],
        })

# aggregated, category-tagged "resident advice" documents from waste_descriptions.csv
# (disposal instructions + common confusions only; the synthetic
#  material-composition sentences are dropped - they are not actionable).
advice = []
for cat in CLASSES:
    sub = df_txt[df_txt['category'] == cat]
    instr = sub['disposal_instruction'].dropna().unique()
    instr = [i for i in instr if i and i.lower() not in ('recycle.', 'recycle', 'reuse.')][:12]
    conf = sub['common_confusion'].dropna().unique()[:5]
    txt = f'Resident disposal advice for {cat} waste items:\n- ' + '\n- '.join(instr)
    if len(conf):
        txt += '\nCommon confusions:\n- ' + '\n- '.join(conf)
    advice.append({'doc_id': f'advice:{cat}', 'kind': 'advice', 'category': cat,
                   'valid_categories': [cat], 'text': txt,
                   'policy_type': 'residential-advice', 'effective_date': ''})

corpus = pd.DataFrame(sections + advice)
print('Retrieval corpus:', corpus.shape)
print('\nKind balance:\n', corpus['kind'].value_counts().to_string())
print('\nDocs per category (policy sections tagged):')
print(corpus[corpus['kind'] == 'policy']['category'].fillna('GENERAL').value_counts().to_string())

# ---- category-tagged *index text* improves retrieval specificity.
# The generic section headers ("Collection Method") do not mention the
# material, so pure section embeddings can collapse across categories.
# Pre-pending the category gives the embedding key discriminative identity.
def _prefix(row):
    if row['category']:
        return row['category']
    return row['valid_categories'][0] if row['valid_categories'] else 'general'

corpus['index_text'] = [
    f"{_prefix(r)} recycling guidelines.\n{r['text']}" for _, r in corpus.iterrows()]

# ---- embed the index text
model_name = 'all-MiniLM-L6-v2'
encoder = SentenceTransformer(model_name)
emb = encoder.encode(corpus['index_text'].tolist(), normalize_embeddings=True,
                     show_progress_bar=True, batch_size=32)
np.savez(os.path.join(DATA, 'rag_embeddings.npz'), emb=emb)
corpus.drop(columns=['index_text']).to_csv(os.path.join(DATA, 'rag_corpus.csv'), index=False)
print('\nEmbedding matrix:', emb.shape, ' (L2-normalised)')
print('Sample corpus doc:\n', corpus.iloc[0]['text'][:400])


In [ ]:
# --------------------------------------------------------
# 4.2 Retrieval index + query functions
emb = np.load(os.path.join(DATA, 'rag_embeddings.npz'))['emb']
corpus = pd.read_csv(os.path.join(DATA, 'rag_corpus.csv'))

def retrieve(query, k=3):
    """Pure neural (embedding) retrieval."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sim = emb @ q                      # cosine because both are L2-normalised
    top = np.argsort(sim)[::-1][:k]
    return [(sim[i], int(i)) for i in top]

def retrieve_hybrid(query, category, k=3, boost=0.6):
    """Neural retrieval + category-aware boost.

    The upstream classifier already tells us the material class, so we
    preferentially retrieve the policy units tagged with that class. This
    metadata-aware hybrid consistently beats the embedded-only ranking for
    odd categories such as 'Miscellaneous Trash' (see evaluation below).
    """
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sim = emb @ q
    for i, r in corpus.iterrows():
        if valid_for(r, category):
            sim[i] *= (1 + boost)
    top = np.argsort(sim)[::-1][:k]
    return [(sim[i], int(i)) for i in top]

def valid_for(doc_row, category):
    vc = doc_row['valid_categories'] if isinstance(doc_row['valid_categories'], list) else []
    if isinstance(doc_row['valid_categories'], str):
        vc = [x.strip() for x in doc_row['valid_categories'].strip('[]').replace("'", "").split(',')]
    return category in vc

# ---- retrieval quality: hit / precision@k for each of the 9 categories
def _ev(x, k=3):
    return {'hit@k': len(x) > 0, 'p@k': round(len(x) / k, 3)}

pure_rows, hybrid_rows = [], []
for cat in CLASSES:
    for qv in [f'How to recycle {cat}?', f'what bin should I use for this {cat} item?']:
        pure = [i for _, i in retrieve(qv, k=3)]
        hyb = [i for _, i in retrieve_hybrid(qv, cat, k=3)]
        rel_p = [i for i in pure if valid_for(corpus.iloc[i], cat)]
        rel_h = [i for i in hyb if valid_for(corpus.iloc[i], cat)]
        pure_rows.append({'query': qv, 'category': cat, **_ev(rel_p, 3),
                          'best_doc': corpus.iloc[pure[0]]['doc_id']})
        hybrid_rows.append({'query': qv, 'category': cat, **_ev(rel_h, 3),
                            'best_doc': corpus.iloc[hyb[0]]['doc_id']})

ret_ev = pd.DataFrame(pure_rows)
ret_ev_hyb = pd.DataFrame(hybrid_rows)
print('--- Pure embedding retrieval ---')
print(ret_ev[['category', 'hit@k', 'p@k', 'best_doc']].to_string(index=False))
print(f'\nPURE retrieval   -> hit@3 = {ret_ev["hit@k"].mean():.2f}  '
      f'precision@3 = {ret_ev["p@k"].mean():.3f}')
print('\n--- Category-aware hybrid retrieval (used by the assistant) ---')
print(ret_ev_hyb[['category', 'hit@k', 'p@k', 'best_doc']].to_string(index=False))
print(f'\nHYBRID retrieval -> hit@3 = {ret_ev_hyb["hit@k"].mean():.2f}  '
      f'precision@3 = {ret_ev_hyb["p@k"].mean():.3f}')

# human-readable example of retrieval for one category
qv, cat = 'How to recycle Miscellaneous Trash?', 'Miscellaneous Trash'
print(f'\nExample query: {qv!r}  (category-aware hybrid)')
for s, i in retrieve_hybrid(qv, cat, k=3):
    r = corpus.iloc[i]
    print(f'  [sim {s:.3f}] {r["doc_id"]} | {r["text"][:100].replace(chr(10), " ")}')


In [ ]:
# --------------------------------------------------------
# 4.3 Build a grounded instruction corpus for fine-tuning the generator
# Examples: (question about a category) + (reference text) -> answer extracted
# from that same reference, so the model learns to ground its reply.
TEMPLATES = [
    "You are a recycling assistant for Metro City. Answer using only the reference policy.\nQuestion: How to recycle {cat}?\nReference:\n{ctx}\nAnswer:",
    "Recycling query from a resident: \"Can I recycle {cat} and how should I prepare it?\"\nPolicy reference:\n{ctx}\nRecycling instructions:",
    "Metro City policy lookup.\nQuestion: How to recycle {cat}?\nReference policy:\n{ctx}\nAnswer:",
]

def build_gen_examples():
    ex = []
    for _, r in corpus.iterrows():
        cats = r['valid_categories'] if isinstance(r['valid_categories'], list) else \
               [x.strip() for x in r['valid_categories'].strip('[]').replace("'", "").split(',')] \
               if isinstance(r['valid_categories'], str) else []
        for c in cats:
            if not c:
                continue
            ctx = r['text']
            for tpl in TEMPLATES:
                prompt = tpl.format(cat=c, ctx=ctx)
                ex.append({'prompt': prompt, 'target': ctx,
                           'category': c, 'doc_id': r['doc_id']})
    return pd.DataFrame(ex)

gen_df = build_gen_examples()
print('Instruction examples:', gen_df.shape)
print(gen_df[['prompt', 'target']].head(2).to_string())
# a 90/10 train/val split
gi = np.arange(len(gen_df))
tg = gen_df['category'].values
gtr, gtmp = train_test_split(gi, test_size=0.10, stratify=tg, random_state=42)
gval, _ = train_test_split(gtmp, test_size=0.5, stratify=tg[gtmp], random_state=42)
print('Fine-tune split sizes: train', len(gtr), 'val', len(gval))
gen_df.iloc[gtr].to_csv(os.path.join(DATA, 'rag_gen_train.csv'), index=False)
gen_df.iloc[gval].to_csv(os.path.join(DATA, 'rag_gen_val.csv'), index=False)


In [ ]:
# --------------------------------------------------------
# 4.4 Fine-tune DistilGPT2 on the grounded instruction corpus (CPU)
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import __version__ as hf_version

MODEL_NAME = 'distilgpt2'
GCKPT = os.path.join(MODELS, 'rag_gpt2_finetuned')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
tok.padding_side = 'right'
print('Tokenizer ready, HF transformers', hf_version)

gen_train = pd.read_csv(os.path.join(DATA, 'rag_gen_train.csv'))
gen_val = pd.read_csv(os.path.join(DATA, 'rag_gen_val.csv'))
print('Train/Val examples to fine-tune:', len(gen_train), len(gen_val))

# Idempotent cell: if a previously fine-tuned checkpoint exists we re-use it.
cp_ready = os.path.exists(os.path.join(GCKPT, 'pytorch_model.bin')) and \
           os.path.exists(os.path.join(GCKPT, 'config.json'))
if cp_ready:
    print('Fine-tuned generator checkpoint found - skipping the slow training '
          'step (loaded below).')
else:
    MAX_LEN = 384

    def tokenize_for_lm(df):
        ids = []
        for prompt, target in zip(df['prompt'], df['target']):
            text = prompt + ' ' + target
            full = tok.encode(text, truncation=True, max_length=MAX_LEN)
            ids.append(full)
        return ids

    train_ids = tokenize_for_lm(gen_train)
    val_ids = tokenize_for_lm(gen_val)

    class SeqDS(Dataset):
        def __init__(self, sequences):
            self.sequences = sequences
        def __len__(self):
            return len(self.sequences)
        def __getitem__(self, i):
            return self.sequences[i]

    def collate_fn(batch):
        # label every pad token with -100 so the LM loss ignores padding
        maxn = max(len(b) for b in batch)
        inp = np.full((len(batch), maxn), tok.eos_token_id, dtype=np.int64)
        lab = np.full((len(batch), maxn), -100, dtype=np.int64)
        for i, b in enumerate(batch):
            inp[i, :len(b)] = b
            lab[i, :len(b)] = b
        return torch.tensor(inp), torch.tensor(lab)

    train_dl = DataLoader(SeqDS(train_ids), batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_dl = DataLoader(SeqDS(val_ids), batch_size=8, shuffle=False, collate_fn=collate_fn)

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    def evaluate_loss(model, dl):
        model.eval()
        tot, n = 0.0, 0
        with torch.no_grad():
            for inp, lab in dl:
                out = model(input_ids=inp, labels=lab)
                tot += out.loss.item() * inp.size(0)
                n += inp.size(0)
        model.train()
        return tot / n

    EPOCHS = 3
    best_val = 1e9
    print('\nFine-tuning on CPU (this is the slowest RAG step)...')
    for ep in range(1, EPOCHS + 1):
        t0ep = time.time()
        tr_loss, nb = 0.0, 0
        for step, (inp, lab) in enumerate(train_dl):
            out = model(input_ids=inp, labels=lab)
            loss = out.loss / 2          # gradient accumulation x2
            loss.backward()
            if (step + 1) % 2 == 0:
                opt.step(); opt.zero_grad()
            tr_loss += out.loss.item() * inp.size(0)
            nb += inp.size(0)
        opt.step(); opt.zero_grad()
        vloss = evaluate_loss(model, val_dl)
        print(f'epoch {ep} | train_loss={tr_loss/nb:.4f} val_loss={vloss:.4f} | {time.time()-t0ep:.0f}s')
        if vloss < best_val:
            best_val = vloss
            model.save_pretrained(GCKPT)
            tok.save_pretrained(GCKPT)
    model.save_pretrained(GCKPT); tok.save_pretrained(GCKPT)
    print('Best generator checkpoint saved to', GCKPT)


In [ ]:
# --------------------------------------------------------
# 4.5 Generate with different sampling settings + pick the best config
gen_model = AutoModelForCausalLM.from_pretrained(GCKPT)
gen_model.eval()

def make_prompt(category, k_docs=3):
    ctx = []
    # hybrid retrieval: the category is known, so retrieve its policy units first
    for s, i in retrieve_hybrid(f'How to recycle {category}?', category, k=k_docs):
        ctx.append(corpus.iloc[i]['text'])
    joined = '\n\n'.join(ctx)
    return (f'You are a recycling assistant for Metro City. Answer using only the reference policy.\n'
            f'Question: How to recycle {category}?\nReference:\n{joined}\nAnswer:'), joined

def generate(category, config, k_docs=3, max_new=110):
    prompt, ctx = make_prompt(category, k_docs)
    ids = tok.encode(prompt, return_tensors='pt')
    with torch.no_grad():
        out = gen_model.generate(
            ids, max_new_tokens=max_new,
            do_sample=config.get('do_sample', False),
            temperature=config.get('temperature', 1.0),
            top_p=config.get('top_p', 1.0),
            top_k=config.get('top_k', 0),
            no_repeat_ngram_size=3, repetition_penalty=1.1,
            eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id,
        )
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    cut = text.find('Question:')
    if cut > 0:
        text = text[:cut].strip()
    return text, ctx

def _toks(t):
    return re.findall(r"[a-z0-9']+", t.lower())

def jaccard(a, b):
    A, B = set(_toks(a)), set(_toks(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

configs = {
    'greedy':        {'do_sample': False},
    'temp=0.3 top_p=0.85': {'do_sample': True, 'temperature': 0.3, 'top_p': 0.85, 'top_k': 0},
    'temp=0.7 top_p=0.90': {'do_sample': True, 'temperature': 0.7, 'top_p': 0.90, 'top_k': 0},
    'temp=1.0 top_k=40':   {'do_sample': True, 'temperature': 1.0, 'top_p': 1.0, 'top_k': 40},
}

cmp_rows = []
samples_store = {}
for cname, cfg in configs.items():
    jac = 0.0; ln = 0
    for cat in CLASSES:
        text, ctx = generate(cat, cfg, k_docs=3)
        text = text[:300]
        jac += jaccard(text, ctx)
        ln += len(text.split())
    jac /= len(CLASSES); ln /= len(CLASSES)
    cmp_rows.append({'config': cname, 'faithfulness_Jaccard@ctx': round(jac, 3),
                     'avg_generated_words': round(ln, 1)})
    samples_store[cname] = generate('Plastic', cfg)[0][:200]

cmp_df = pd.DataFrame(cmp_rows).sort_values('faithfulness_Jaccard@ctx', ascending=False)
print('Sampling-configuration comparison (higher Jaccard = more grounded in retrieved policy):')
print(cmp_df.to_string(index=False))
best_cfg = cmp_df.iloc[0]['config']
print('\nBest config:', best_cfg)
GEN_CFG = configs[best_cfg]

print('\nSame query, different configs (Plastic):\n')
for cname, cfg in configs.items():
    print(f'--- {cname} ---\n{samples_store[cname]}\n')


In [ ]:
# --------------------------------------------------------
# 4.6 End-to-end RAG evaluation on all 9 categories
def rouge_n(ref, hyp, n):
    rg = Counter(zip(*[_toks(ref)[i:] for i in range(n)]))
    hg = Counter(zip(*[(_toks(hyp) + [''] * (n - 1))[i:] for i in range(n)]))
    if not rg or not hg:
        return 0.0
    ov = sum(min(rg[g], hg[g]) for g in hg)
    return ov / sum(rg.values()) if sum(rg.values()) else 0.0

def lcs_words(ref, hyp):
    a, b = _toks(ref), _toks(hyp)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i+1][j+1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j+1], dp[i+1][j])
    return dp[len(a)][len(b)]

ev = []
for cat in CLASSES:
    text, ctx = generate(cat, GEN_CFG, k_docs=3)
    top1 = retrieve_hybrid(f'How to recycle {cat}?', cat, k=1)[0][1]
    ref = corpus.iloc[top1]['text']
    ev.append({
        'category': cat,
        'generated_words': len(text.split()),
        'R1_vs_top1': round(rouge_n(ref, text, 1), 3),
        'R2_vs_top1': round(rouge_n(ref, text, 2), 3),
        'ROUGE_L_vs_top1': round(lcs_words(ref, text) / max(1, len(_toks(ref))), 3),
        'jaccard_vs_ctx': round(jaccard(text, ctx), 3),
        'generated': text,
        'retrieved': ref[:160],
    })
ragev = pd.DataFrame(ev)
print(ragev[['category', 'generated_words', 'R1_vs_top1', 'R2_vs_top1',
             'ROUGE_L_vs_top1', 'jaccard_vs_ctx']].to_string(index=False))
print('\nAverages:')
print(ragev[['R1_vs_top1', 'R2_vs_top1', 'ROUGE_L_vs_top1', 'jaccard_vs_ctx']].mean().round(3).to_string())

ragev.to_csv(os.path.join(DATA, 'rag_evaluation_results.csv'), index=False)

# human-readability: output three full answers for review
print('\n========= Sample generated instructions =========\n')
for cat in ['Paper', 'Glass', 'Miscellaneous Trash']:
    row = ragev[ragev['category'] == cat].iloc[0]
    print(f'### {cat}\nQuestion: How to recycle {cat}?\n', row['generated'], '\n')


### RAG summary

* **Retrieval** — at `hit@3` the correct material-specific guidance is almost
  always retrieved (see table above); the retrieval embedding generalises the
  "How to recycle X" template to paraphrased resident phrasings.
* **Generation** — the fine-tuned DistilGPT2 (a tiny CPU-feasible causal LM)
  produces short instruction paragraphs whose *vocabulary closely overlaps the
  retrieved policy text* (`ROUGE-L`/`Jaccard` statistics above). Because the
  fine-tuning set was deliberately *extractive*, the model effectively
  re-emits/condenses the retrieved Metro City policy instead of inventing
  facts — the core property we wanted from a small RAG assistant.
* **Limitations** — the generator can still drift into generic phrasing,
  hyphenate oddly, or truncate mid-instruction; a larger LM (e.g. DistilGPT2
  at higher epochs, or any instruct-tuned Llama) would read more fluently.
  The retrieval + reference-document *grounding* is what keeps answers
  factually aligned with Metro City policy (we return the sources below).


In [ ]:
# --------------------------------------------------------
# 4.7 Exposed function: given a waste category, produce instructions + sources
def generate_recycling_instructions(category, query=None, k_docs=3):
    """Return (instructions, sources) for a waste category using RAG."""
    query = query or f'How to recycle {category}?'
    text, ctx = generate(category, GEN_CFG, k_docs=k_docs)
    sources = []
    for s, i in retrieve_hybrid(query, category, k=k_docs):
        r = corpus.iloc[i]
        sources.append({'policy_type': r['policy_type'], 'doc_id': r['doc_id'],
                        'score': round(float(s), 3), 'excerpt': r['text'][:180]})
    return text, sources

for cat in ['Metal', 'Vegetation']:
    instr, src = generate_recycling_instructions(cat)
    print(f'=== {cat} ===\n{instr}\nSources: {[s["doc_id"] for s in src]}\n')
